In [41]:
import urllib.request, urllib.parse, json, time
import pandas as pd, numpy as np
from pathlib import Path

# 47 lokasyonun koordinatları (Open-Meteo geocoding ile)
KOORDINAT = {
    'AHMETLİ': (38.5196, 27.9386),   'AKHİSAR': (38.9185, 27.8401),
    'ALAŞEHİR': (38.3508, 28.5172),  'ALİAĞA': (38.7998, 26.9720),
    'BALÇOVA': (38.3920, 27.0426),   'BAYINDIR': (38.2174, 27.6474),
    'BAYRAKLI': (38.4672, 27.1638),  'BERGAMA': (39.1207, 27.1805),
    'BEYDAĞ': (38.0859, 28.2084),    'BORNOVA': (38.4792, 27.2399),
    'BUCA': (38.3983, 27.1666),      'DEMİRCİ': (39.0461, 28.6589),
    'DİKİLİ': (39.0710, 26.8902),    'FOÇA': (38.6703, 26.7566),
    'GAZİEMİR': (38.3239, 27.1292),  'GÖLMARMARA': (38.7139, 27.9142),
    'GÖRDES': (38.9328, 28.2894),    'GÜZELBAHÇE': (38.3604, 26.8865),
    'KARABAĞLAR': (38.3820, 27.1320),'KARABURUN': (38.6364, 26.5109),
    'KARŞIYAKA': (38.4577, 27.1142), 'KEMALPAŞA': (38.4262, 27.4173),
    'KINIK': (39.0872, 27.3833),     'KIRKAĞAÇ': (39.1064, 27.6693),
    'KONAK': (38.4000, 27.1000),     'KULA': (38.5473, 28.6498),
    'KÖPRÜBAŞI': (38.7497, 28.4047), 'KİRAZ': (38.2306, 28.2044),
    'MENDERES': (38.2496, 27.1343),  'MENEMEN': (38.6075, 27.0694),
    'NARLIDERE': (38.3939, 27.0128), 'SALİHLİ': (38.4826, 28.1477),
    'SARIGÖL': (38.2395, 28.6966),   'SARUHANLI': (38.7345, 27.5681),
    'SEFERİHİSAR': (38.1975, 26.8388),'SELENDİ': (38.7444, 28.8678),
    'SELÇUK': (37.9514, 27.3685),    'SOMA': (39.1855, 27.6094),
    'TORBALI': (38.1820, 27.3350),   'TURGUTLU': (38.4953, 27.6997),
    'TİRE': (38.0888, 27.7351),      'URLA': (38.3229, 26.7640),
    'YUNUSEMRE': (38.6150, 27.3900), 'ÇEŞME': (38.3261, 26.3057),
    'ÇİĞLİ': (38.4965, 27.0703),     'ÖDEMİŞ': (38.2278, 27.9696),
    'ŞEHZADELER': (38.6280, 27.4400),
}

DOSYA = Path('../data/external/hava_durumu.csv')


if DOSYA.exists(): # Daha önce çekildiyse kontrol
    hava = pd.read_csv(DOSYA, parse_dates=['tarih'])
    print('önbellekten okundu:', hava.shape)
else:
    parcalar = []
    for i, (yer, (enlem, boylam)) in enumerate(KOORDINAT.items(), 1):
        url = "https://archive-api.open-meteo.com/v1/archive?" + urllib.parse.urlencode({
            'latitude': enlem, 'longitude': boylam,
            'start_date': '2025-01-01', # train'in ilk günü
            'end_date':   '2026-07-31', # test'in son günü
            'daily': 'temperature_2m_mean,temperature_2m_max,temperature_2m_min',
            'timezone': 'Europe/Istanbul',
        })
        with urllib.request.urlopen(url, timeout=60) as r:
            d = json.load(r)
        parcalar.append(pd.DataFrame({
            'yer': yer,
            'tarih': pd.to_datetime(d['daily']['time']),
            'sic_ort': d['daily']['temperature_2m_mean'],
            'sic_max': d['daily']['temperature_2m_max'],
            'sic_min': d['daily']['temperature_2m_min'],
        }))
        print(f"  [{i}/{len(KOORDINAT)}] {yer}", end='\r')
        time.sleep(0.3)       

    hava = pd.concat(parcalar, ignore_index=True)
    DOSYA.parent.mkdir(parents=True, exist_ok=True)
    hava.to_csv(DOSYA, index=False)
    print('\nçekildi ve kaydedildi:', hava.shape)

#Kontrolleri sağla
print('lokasyon sayısı:', hava['yer'].nunique())
print('tarih aralığı:', hava['tarih'].min().date(), '->', hava['tarih'].max().date())
print('eksik sıcaklık:', hava['sic_ort'].isna().sum())
print('Temmuz 2026 ortalama sıcaklık (en sıcak 10 yer):')
tem = hava[(hava['tarih'] >= '2026-07-01') & (hava['tarih'] <= '2026-07-31')]
print(tem.groupby('yer')['sic_ort'].mean().sort_values(ascending=False).head(10).round(1).to_string())

önbellekten okundu: (27119, 5)
lokasyon sayısı: 47
tarih aralığı: 2025-01-01 -> 2026-07-31
eksik sıcaklık: 0
Temmuz 2026 ortalama sıcaklık (en sıcak 10 yer):
yer
KARŞIYAKA     29.2
KARABAĞLAR    29.1
SALİHLİ       29.0
SELÇUK        28.9
ÖDEMİŞ        28.9
TORBALI       28.8
KONAK         28.7
BORNOVA       28.6
TİRE          28.6
AHMETLİ       28.6


In [42]:
import lightgbm as lgb

RAW = '../data/raw/'
train = pd.read_csv(RAW + 'train.csv')
test  = pd.read_csv(RAW + 'test.csv')
train['tarih'] = pd.to_datetime(train['tarih'])
test['tarih']  = pd.to_datetime(test['tarih'])


def rmsle(gercek, tahmin):
    return np.sqrt(np.mean((np.log1p(tahmin) - np.log1p(gercek))**2))


# EDA bulgumuz vardı: 444 satır guc x 24 saat sınırı (fiziksel imkansız) aşıyor.
# 23 trafo lokalize olduğundan trafo bazlı medyan ile dolduruldu.
train['asiri_oran'] = train['tuketim'] / (train['guc'] * 24)
train['tuketim_duzeltilmis'] = train['tuketim']
for t in train.loc[train['asiri_oran'] > 2, 'tanim'].unique():
    medyan = train.loc[(train['tanim'] == t) & (train['asiri_oran'] <= 2), 'tuketim'].median()
    train.loc[(train['tanim'] == t) & (train['asiri_oran'] > 2), 'tuketim_duzeltilmis'] = medyan

TATILLER = pd.to_datetime([
    '2025-01-01','2025-03-29','2025-03-30','2025-03-31','2025-04-01','2025-04-23',
    '2025-05-01','2025-05-19','2025-06-05','2025-06-06','2025-06-07','2025-06-08','2025-06-09',
    '2025-08-30','2025-10-29',
    '2026-01-01','2026-03-19','2026-03-20','2026-03-21','2026-04-23','2026-05-01','2026-05-19',
    '2026-05-26','2026-05-27','2026-05-28','2026-05-29','2026-05-30','2026-08-30','2026-10-29',
]).date

GUC_BINS = [0, 100, 250, 500, 1000, 2000, 5000, np.inf]


def temel_ozellikler(df):
    df = df.copy()
    lok = df['lokasyon'].str.split('>', expand=True)
    df['il'] = lok[0]
    df['bolge'] = lok[1]
    df['ilce'] = (lok[2] if lok.shape[1] > 2 else pd.Series(np.nan, index=df.index)).fillna('YOK')

    df['ay'] = df['tarih'].dt.month
    df['gun'] = df['tarih'].dt.day
    df['haftanin_gunu'] = df['tarih'].dt.dayofweek
    df['hafta_sonu'] = (df['haftanin_gunu'] >= 5).astype(int)
    df['yilin_gunu'] = df['tarih'].dt.dayofyear
    df['tatil'] = df['tarih'].dt.date.isin(TATILLER).astype(int)

    df['guc_bucket'] = pd.cut(df['guc'], bins=GUC_BINS)
    df['guc_bucket_str'] = df['guc_bucket'].astype(str)
    return df


train = temel_ozellikler(train)
test  = temel_ozellikler(test)

KATEGORIK = ['il', 'bolge', 'ilce', 'guc_bucket_str']

print('train', train.shape, '| test', test.shape)
print('düzeltilen satır sayısı:', (train['tuketim'] != train['tuketim_duzeltilmis']).sum())
print('train tarih:', train['tarih'].min().date(), '->', train['tarih'].max().date())
print('test tarih:', test['tarih'].min().date(), '->', test['tarih'].max().date())

train (1226237, 18) | test (714688, 16)
düzeltilen satır sayısı: 444
train tarih: 2025-01-01 -> 2026-03-31
test tarih: 2026-04-01 -> 2026-07-31


In [43]:
hava = pd.read_csv('../data/external/hava_durumu.csv', parse_dates=['tarih'])
hava = hava.sort_values(['yer', 'tarih']).reset_index(drop=True)

# CDD (Cooling Degree Day): 22 derece üstü her derece = klima yükü olarak adlandır
# HDD (Heating Degree Day): 18 derece üstü her derece = ısıtma yükü olarak adlandır
# İki sütun tüketim sıcaklıkla linear değil V shape var, doğru ve kolay öğrenilmesi için iki sütun.
hava['cdd'] = (hava['sic_ort'] - 22).clip(lower=0)
hava['hdd'] = (18 - hava['sic_ort']).clip(lower=0)

# Günlük fark büyükse kara iklimde(iç), küçükse deniz etkisi(dış)
hava['gunluk_fark'] = hava['sic_max'] - hava['sic_min']

# Termal atalet case: bina vs tek günde soğuyup ısınmaz, önceki günler de önemli
hava['sic_3gun'] = hava.groupby('yer')['sic_ort'].transform(lambda s: s.rolling(3, min_periods=1).mean())
hava['cdd_3gun'] = hava.groupby('yer')['cdd'].transform(lambda s: s.rolling(3, min_periods=1).mean())

HAVA_KOLONLARI = ['sic_ort', 'sic_max', 'sic_min', 'cdd', 'hdd', 'gunluk_fark', 'sic_3gun', 'cdd_3gun']


def hava_ekle(df):
    df = df.copy()
    # Manisada ilçe kırılım yok: bölge adını kullanıyorum.
    df['yer'] = np.where(df['ilce'] != 'YOK', df['ilce'], df['bolge'])
    return df.merge(hava[['yer', 'tarih'] + HAVA_KOLONLARI], on=['yer', 'tarih'], how='left')

train = hava_ekle(train)
test  = hava_ekle(test)

print('eksik sıcaklıklar: train:', train['sic_ort'].isna().sum(), '| test:', test['sic_ort'].isna().sum())
print()
print('yilin_gunu vs sic_ort: hangisi egitim/test arasında örtüşüyor')
print('\nyilin_gunu için train:', train['yilin_gunu'].min(), '->', train['yilin_gunu'].max(), '| test:', test['yilin_gunu'].min(), '->', test['yilin_gunu'].max())
print('\nsic_ort için train:', round(train['sic_ort'].min(), 1), '->', round(train['sic_ort'].max(), 1), '| test:', round(test['sic_ort'].min(), 1), '->', round(test['sic_ort'].max(), 1))
print()
print('\nAylık sıcaklık tüketimle ilişkili mi')
print(train.groupby('ay').agg(
      sicaklik=('sic_ort', 'mean'), 
      tuketim=('tuketim_duzeltilmis', 'median')).round(1).to_string())

eksik sıcaklıklar: train: 0 | test: 0

yilin_gunu vs sic_ort: hangisi egitim/test arasında örtüşüyor

yilin_gunu için train: 1 -> 365 | test: 91 -> 212

sic_ort için train: -5.5 -> 36.4 | test: 3.1 -> 32.6


Aylık sıcaklık tüketimle ilişkili mi
    sicaklik  tuketim
ay                   
1        9.1   1102.9
2        8.7   1084.2
3       11.5    994.2
4       14.3    877.7
5       20.3    821.9
6       26.9   1196.7
7       30.2   1683.3
8       28.5   1502.3
9       24.4   1096.6
10      17.2    831.5
11      15.4    882.1
12       9.1   1090.4


In [44]:
def trafo_profili_olustur(gecmis, origin_tarihi):
    g = gecmis[gecmis['tarih'] < origin_tarihi].copy()
    g['log_t'] = np.log1p(g['tuketim_duzeltilmis'])

    profil = g.groupby('tanim').agg(
        trafo_ortalama   = ('tuketim_duzeltilmis', 'mean'),
        trafo_medyan     = ('tuketim_duzeltilmis', 'median'),
        trafo_gun_sayisi = ('tuketim_duzeltilmis', 'count'),
        log_ortalama     = ('log_t', 'mean'),
        log_medyan       = ('log_t', 'median'),
        log_std          = ('log_t', 'std'),
        sifir_orani      = ('tuketim_duzeltilmis', lambda x: (x == 0).mean()),
    ).reset_index()

    for pencere in [30, 90]:
        son  = g[g['tarih'] >= origin_tarihi - pd.Timedelta(days=pencere)]
        ozet = son.groupby('tanim').agg(**{
            f'son{pencere}_log_ort': ('log_t', 'mean'),
            f'son{pencere}_sifir'  : ('tuketim_duzeltilmis', lambda x: (x == 0).mean()),
        }).reset_index()
        profil = profil.merge(ozet, on='tanim', how='left')

    profil['log_trend'] = profil['son30_log_ort'] - profil['son90_log_ort']
    return profil


def trafo_gun_profili(gecmis, origin_tarihi):
    g = gecmis[gecmis['tarih'] < origin_tarihi][['tanim', 'haftanin_gunu', 'tuketim_duzeltilmis']].copy()
    ort = g.groupby('tanim')['tuketim_duzeltilmis'].mean().rename('_ort')
    g = g.merge(ort, on='tanim', how='left')
    g['oran'] = g['tuketim_duzeltilmis'] / g['_ort'].replace(0, np.nan)
    return g.groupby(['tanim', 'haftanin_gunu'])['oran'].mean().rename('trafo_gun_orani').reset_index()


#Tek fonk: eğitim/validation/test'te aynı şekilde kullan. Sadece origini değiştir eğitim ve tahmin koşulları birebir aynı sızıntı olmaz
def veri_seti_kur(gecmis, hedef_satirlar, origin_tarihi, k=30):
    g = gecmis[gecmis['tarih'] < origin_tarihi].copy()
    g['log_t'] = np.log1p(g['tuketim_duzeltilmis'])
    genel_log  = g['log_t'].mean()

    # shrinkage (k=30): küçük grupları genel ortalamaya çek
    grup = g.groupby(['il', 'bolge', 'guc_bucket'], observed=True)['log_t'].agg(['sum', 'count']).reset_index()
    grup['grup_log_ort'] = (grup['sum'] + k * genel_log) / (grup['count'] + k)
    il_guc = g.groupby(['il', 'guc_bucket'], observed=True)['log_t'].mean().rename('ilguc_log_ort').reset_index()

    df = hedef_satirlar.drop(columns=['grup_log_ort', 'ilguc_log_ort'], errors='ignore').copy()
    df = df.merge(grup[['il', 'bolge', 'guc_bucket', 'grup_log_ort']],
                  on=['il', 'bolge', 'guc_bucket'], how='left')
    df = df.merge(il_guc, on=['il', 'guc_bucket'], how='left')
    
    # iki kademeli yedek: il+bolge+guc -> il+guc -> genel ortalama
    df['grup_log_ort'] = df['grup_log_ort'].fillna(df['ilguc_log_ort']).fillna(genel_log)

    df = df.merge(trafo_profili_olustur(gecmis, origin_tarihi), on='tanim', how='left')
    df = df.merge(trafo_gun_profili(gecmis, origin_tarihi), on=['tanim', 'haftanin_gunu'], how='left')
    df['ufuk_gun'] = (df['tarih'] - origin_tarihi).dt.days
    return df


def origin_seti(hedef_bas, hedef_bit, origin):
    hedef = train[(train['tarih'] >= hedef_bas) & (train['tarih'] <= hedef_bit)]
    return veri_seti_kur(train, hedef, pd.Timestamp(origin))


TEMEL_OZELLIKLER = [
    'guc', 'ay', 'gun', 'haftanin_gunu', 'hafta_sonu', 'yilin_gunu', 'tatil',
    'il', 'bolge', 'ilce', 'guc_bucket_str', 'grup_log_ort',
    'trafo_ortalama', 'trafo_medyan', 'trafo_gun_sayisi',
    'log_ortalama', 'log_medyan', 'log_std', 'sifir_orani',
    'son30_log_ort', 'son90_log_ort', 'son30_sifir', 'son90_sifir',
    'log_trend', 'trafo_gun_orani',
]
print('temel özellikler:', len(TEMEL_OZELLIKLER), '| hava özellikleri:', len(HAVA_KOLONLARI))

temel özellikler: 25 | hava özellikleri: 8


### Validation'ı değiştirme tercihi:

02_model'de validation Aralık-Mart (kış), test ise Nisan-Temmuz (yaz). Kışta ölçülen hata yazdaki hatayı kapsamıyor: Yerel RMSLE 0.9876 VS Kaggle RMSLE 1.08157 

Yeni kurgu test setinin bir yIl Öncesini birebir taklit ediyor:
- test -> origin 2026-04-01, hedef Nisan-Temmuz 2026, ufuk 0-121 gün
- val -> origin 2025-04-01, hedef Nisan-Temmuz 2025, ufuk 0-121 gün

Eğitim hedefleri validation başlamadan bitiyor (Mart 2025 sonu) (sızıntı yok)
Veride tek yaz var + val yaz sızıntısız imkansız
Model hiç yaz görmeden yazı tahmin ediyordu
Hava durumu mevsim genellemesi katkısı net 

### Eksik kalan parça: cold-start

Mevsimi düzeltmek tek başına yetmedi. Yeni kurgu 1.0281 verdi, Kaggle aynı konfigürasyona
1.08157 dedi. Kalan farkın sebebi: validationda profilsiz trafo oranı %7.5, testte %22.16.
Yani gerçek zorluğun üçte birini ölçüyorduk.

Yeni case keşfedildi: 2026-05-11'de 1326 trafo aynı anda devreye giriyor, günlük satır
sayısı 3928'den 6802'ye çıkıyor. Toplam 2024 yeni trafo var, test satırlarının %22.16'sı.
Hepsi Mayıs-Temmuz aralığında, yani en yüksek tüketimli dönemde. Bu satırlarda trafo
geçmişi yok, elimizde sadece guc + lokasyon var.

Aşağıda validation trafolarının bir kısmını kasıtlı olarak profilsiz bırakıp oranı testle
eşitliyoruz. Ayrıca skoru iki segmentte ayrı raporluyoruz (profilli / profilsiz), çünkü
toplam skor hangi segmentin bozulduğunu gizliyor.


In [45]:
egitim_seti = pd.concat([
    origin_seti('2025-02-01', '2025-03-31', '2025-02-01'),
    origin_seti('2025-03-01', '2025-03-31', '2025-03-01'),
], ignore_index=True)

val_seti = origin_seti('2025-04-01', '2025-07-31', '2025-04-01')

for df in [egitim_seti, val_seti]:
    df['hedef_log'] = np.log1p(df['tuketim_duzeltilmis'])
    for c in KATEGORIK:
        df[c] = df[c].astype('category')

# val kategorileri train sırası ile aynı olmalı LightGBM eşleştirebilmesi için
for c in KATEGORIK:
    val_seti[c] = pd.Categorical(val_seti[c], categories=egitim_seti[c].cat.categories)

print('eğitim', egitim_seti.shape, '| validation', val_seti.shape)
print('eğitim ayları:', sorted(egitim_seti['ay'].unique()))
print('validation ayları:', sorted(val_seti['ay'].unique()))
print('test ayları:', sorted(test['ay'].unique()))
print()
print('val ufku :', val_seti['ufuk_gun'].min(), '-', val_seti['ufuk_gun'].max(), 'gun')
print('test ufku:', (test['tarih'] - pd.Timestamp('2026-04-01')).dt.days.min(),'-', (test['tarih'] - pd.Timestamp('2026-04-01')).dt.days.max(), 'gun')
print()

#     **Cold-start: validationu testle aynı zorluğa getir**
# train'de hiç yok, satır bazında %22.16. Validationda bu oran %7.5 kalıyor. Rastgele trafo seçip profil sütunlarını NaN yaparak oranı eşitledik.
PROFIL_KOLONLARI = ['trafo_ortalama', 'trafo_medyan', 'trafo_gun_sayisi',
                    'log_ortalama', 'log_medyan', 'log_std', 'sifir_orani',
                    'son30_log_ort', 'son90_log_ort', 'son30_sifir', 'son90_sifir',
                    'log_trend', 'trafo_gun_orani']

TEST_COLD_ORAN = (~test['tanim'].isin(set(train['tanim']))).mean()

#satır bazında hedef orana ulaşana kadar rastgele trafoları profilsiz bırak.
def cold_start_uygula(df, hedef_oran, tohum=42):
    df = df.copy()
    zaten_bos = df['log_ortalama'].isna()
    gerekli = int(len(df) * hedef_oran) - int(zaten_bos.sum())
    if gerekli > 0:
        sayim = df.loc[~zaten_bos].groupby('tanim').size()
        aday = np.random.default_rng(tohum).permutation(sayim.index.to_numpy())

        secilen, toplam = [], 0
        for t in aday:
            if toplam >= gerekli:
                break
            secilen.append(t)
            toplam += sayim[t]
        df.loc[df['tanim'].isin(secilen), PROFIL_KOLONLARI] = np.nan

    df['cold_start'] = df['log_ortalama'].isna()
    return df


print('cold-start oranı -> val (ham):', round(val_seti['log_ortalama'].isna().mean(), 4))
val_seti = cold_start_uygula(val_seti, TEST_COLD_ORAN)
print('cold-start oranı -> test:', round(TEST_COLD_ORAN, 4))
print('cold-start oranı -> val (son):', round(val_seti['cold_start'].mean(), 4))
print('profilsiz trafo sayısı:', val_seti.loc[val_seti['cold_start'], 'tanim'].nunique(), '/', val_seti['tanim'].nunique())

eğitim (188679, 44) | validation (274929, 44)
eğitim ayları: [np.int32(2), np.int32(3)]
validation ayları: [np.int32(4), np.int32(5), np.int32(6), np.int32(7)]
test ayları: [np.int32(4), np.int32(5), np.int32(6), np.int32(7)]

val ufku : 0 - 121 gun
test ufku: 0 - 121 gun

cold-start oranı -> val (ham): 0.075
cold-start oranı -> test: 0.2216
cold-start oranı -> val (son): 0.2218
profilsiz trafo sayısı: 1031 / 2891


In [46]:
def dene(ad, ozellikler, egitim, val, kayit, learning_rate=0.05, num_leaves=63):
    kat = [c for c in KATEGORIK if c in ozellikler]
    model = lgb.LGBMRegressor(
        n_estimators=3000, 
        learning_rate=learning_rate,
        num_leaves=num_leaves, 
        random_state=42, 
        verbose=-1)
    model.fit(
        egitim[ozellikler], egitim['hedef_log'],
        eval_X=val[ozellikler], eval_y=val['hedef_log'],
        eval_metric='rmse', categorical_feature=kat,
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
    )
    tahmin = np.clip(np.expm1(model.predict(val[ozellikler])), 0, None)
    gercek = val['tuketim_duzeltilmis'].to_numpy()

    # Toplam skor hangi segmentin bozulduğunu gizliyor, ikisini ayrı ölçüyoruz
    cs = val['cold_start'].to_numpy()
    skor = rmsle(gercek, tahmin)
    skor_cold = rmsle(gercek[cs], tahmin[cs])
    skor_prof = rmsle(gercek[~cs], tahmin[~cs])

    kayit.append({'ad': ad, 'skor': skor, 'cold': skor_cold, 'profilli': skor_prof,
                  'agac': model.best_iteration_, 'ozellikler': ozellikler,
                  'lr': learning_rate, 'leaves': num_leaves})
    print(f'{ad:30s} RMSLE {skor:.4f}  (profilli {skor_prof:.4f} | cold-start {skor_cold:.4f})'
          f'  agac {model.best_iteration_}, ozellik {len(ozellikler)}')
    return model


TEMEL_GUNSUZ = [f for f in TEMEL_OZELLIKLER if f != 'yilin_gunu']

sonuclar_A = []
dene('D0 referans (02_model seti)', TEMEL_OZELLIKLER, egitim_seti, val_seti, sonuclar_A)
dene('D1 yilin_gunu cikarildi', TEMEL_GUNSUZ, egitim_seti, val_seti, sonuclar_A)
dene('D2 hava durumu eklendi', TEMEL_OZELLIKLER + HAVA_KOLONLARI, egitim_seti, val_seti, sonuclar_A)
dene('D3 hava + yilin_gunu yok', TEMEL_GUNSUZ + HAVA_KOLONLARI, egitim_seti, val_seti, sonuclar_A)

D0 referans (02_model seti)    RMSLE 1.2887  (profilli 0.8614 | cold-start 2.2102)  agac 48, ozellik 25
D1 yilin_gunu cikarildi        RMSLE 1.2862  (profilli 0.8572 | cold-start 2.2093)  agac 51, ozellik 24
D2 hava durumu eklendi         RMSLE 1.2922  (profilli 0.8618 | cold-start 2.2189)  agac 51, ozellik 33
D3 hava + yilin_gunu yok       RMSLE 1.2946  (profilli 0.8644 | cold-start 2.2216)  agac 52, ozellik 32


,num_leaves,63
,learning_rate,0.05
,n_estimators,3000
,random_state,42
,verbose,-1
,boosting_type,'gbdt'
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [ ]:
def tablola(kayit, baslik):
    print(baslik)
    t = pd.DataFrame([{'deney': s['ad'], 'RMSLE': round(s['skor'], 4),
                       'profilli': round(s['profilli'], 4), 'cold': round(s['cold'], 4),
                       'agac': s['agac'], 'ozellik': len(s['ozellikler'])} for s in kayit])
    print(t.sort_values('RMSLE').to_string(index=False))
    return min(kayit, key=lambda s: s['skor'])


en_iyi_A = min(sonuclar_A, key=lambda s: s['skor'])
for lr, leaves in [(0.05, 31), (0.05, 127), (0.03, 63), (0.10, 63)]:
    dene(f'D4 lr={lr} leaves={leaves}', en_iyi_A['ozellikler'], egitim_seti, val_seti, sonuclar_A, learning_rate=lr, num_leaves=leaves)

print()
kazanan_A = tablola(sonuclar_A, 'KURGU A: mevsim genellemesi (egitim Subat-Mart, val Nisan-Temmuz)')

D4 lr=0.05 leaves=31           RMSLE 1.2694  (profilli 0.8559 | cold-start 2.1668)  agac 58, ozellik 24
D4 lr=0.05 leaves=127          RMSLE 1.3015  (profilli 0.8691 | cold-start 2.2333)  agac 50, ozellik 24
D4 lr=0.03 leaves=63           RMSLE 1.2838  (profilli 0.8574 | cold-start 2.2028)  agac 83, ozellik 24
D4 lr=0.1 leaves=63            RMSLE 1.2919  (profilli 0.8596 | cold-start 2.2211)  agac 24, ozellik 24

KURGU A: mevsim genellemesi (egitim Subat-Mart, val Nisan-Temmuz)
                      deney  RMSLE  profilli   cold  agac  ozellik
       D4 lr=0.05 leaves=31 1.2694    0.8559 2.1668    58       24
       D4 lr=0.03 leaves=63 1.2838    0.8574 2.2028    83       24
    D1 yilin_gunu cikarildi 1.2862    0.8572 2.2093    51       24
D0 referans (02_model seti) 1.2887    0.8614 2.2102    48       25
        D4 lr=0.1 leaves=63 1.2919    0.8596 2.2211    24       24
     D2 hava durumu eklendi 1.2922    0.8618 2.2189    51       33
   D3 hava + yilin_gunu yok 1.2946    0.8644 2.2

### Kurgu B kaldırıldı: neden yanılttı

Kurgu B aynı dönemi trafo bazında bölüyordu, yani model Temmuz 2025'i eğitimde görüp
Temmuz 2025'i tahmin ediyordu. Hava durumunu "faydalı" (0.9219 -> 0.9194) buldu ve final
modeli buna göre seçtik: sonuç Kaggle'da 1.09015, sub_02'nin 1.08157'sinden kötü.

Hatanın kaynağı şu: Kurgu B yıl genellemesini hiç sınamıyor. Gerçek görevde Temmuz
2025'ten öğrenip Temmuz 2026'yı tahmin etmemiz gerekiyor ve iki yazın sıcaklık deseni
farklı. Kurgu B'de model o yazın kendine özgü sıcaklık dalgalanmalarını doğrudan
ezberleyebiliyordu. Sızıntı tüm özellikleri eşit artırmadı, tam da hava durumu
özelliklerini kolladı.

Kurgu A ise hava durumunu doğru şekilde kötü bulmuştu (1.0281 -> 1.0310). Bundan sonra
tek ölçüt Kurgu A, bunun üzerine de cold-start ekleyeceğiz.





In [ ]:
# Yerel skor 1.2887, Kaggle 1.08157. Ölçüm bu sefer kötüleşti.
# Hipotez: modelin egitim sırasında profilsiz satır görme oranı, validation kurgusunda
# gerçek final egitimden çok daha düşük. Model NaN profili öğrenemiyor.

FINAL_ORIGINLER = [
    ('2025-04-01', '2025-07-31'),
    ('2025-05-01', '2025-08-31'),
    ('2025-06-01', '2025-09-30'),
    ('2025-12-01', '2026-03-31'),
]


def dogal_cold_orani(origin, hedef_bas, hedef_bit, kaynak=None):
    origin = pd.Timestamp(origin)
    gecmis = set(train.loc[train['tarih'] < origin, 'tanim'])
    kaynak = train if kaynak is None else kaynak
    hedef = kaynak[(kaynak['tarih'] >= hedef_bas) & (kaynak['tarih'] <= hedef_bit)]
    return (~hedef['tanim'].isin(gecmis)).mean(), len(hedef)


print('Egitim setlerinde doğal cold-start oranı (profilsiz satır payı)')
print()
print('validation kurgusunun egitim origin-leri:')
for o, b, s in [('2025-02-01', '2025-02-01', '2025-03-31'),
                ('2025-03-01', '2025-03-01', '2025-03-31')]:
    r, n = dogal_cold_orani(o, b, s)
    print(f'origin {o}  hedef {b}..{s}  cold {r:.4f}  satır {n}')

print()
print('final egitim origin-leri:')
for bas, bit in FINAL_ORIGINLER:
    r, n = dogal_cold_orani(bas, bas, bit)
    print(f'  origin {bas}  hedef {bas}..{bit}  cold {r:.4f}  satır {n}')

r_test, n_test = dogal_cold_orani('2026-04-01', '2026-04-01', '2026-07-31', test)
print()
print(f'TEST  origin 2026-04-01  cold {r_test:.4f}  satır {n_test}')
print()

#Sonuç: validation kurgusunun egitimi %1 civarı cold-start görüyor, final egitim %7-14,
#test ise %22. Model profilsiz durumu ne kadar az görürse o segmentte o kadar kötü.


# Hatanın ne kadarı cold-start segmentinden geliyor check
p = val_seti['cold_start'].mean()
k = kazanan_A
pay_cold = p * k['cold']**2
pay_prof = (1 - p) * k['profilli']**2
print(f" Kazanan modelde hata dağılımı (RMSLE^2 payları):")
print(f" profilli segment: satırların %{100*(1-p):.0f}'i, hatanın %{100*pay_prof/(pay_cold+pay_prof):.0f}'i")
print(f" cold-start segment: satırların %{100*p:.0f}'i, hatanın %{100*pay_cold/(pay_cold+pay_prof):.0f}'i")

Egitim setlerinde doğal cold-start oranı (profilsiz satır payı)

validation kurgusunun egitim origin-leri:
origin 2025-02-01  hedef 2025-02-01..2025-03-31  cold 0.0147  satır 123473
origin 2025-03-01  hedef 2025-03-01..2025-03-31  cold 0.0110  satır 65206

final egitim origin-leri:
  origin 2025-04-01  hedef 2025-04-01..2025-07-31  cold 0.0750  satır 274929
  origin 2025-05-01  hedef 2025-05-01..2025-08-31  cold 0.1132  satır 290561
  origin 2025-06-01  hedef 2025-06-01..2025-09-30  cold 0.1156  satır 296798
  origin 2025-12-01  hedef 2025-12-01..2026-03-31  cold 0.1394  satır 444076

TEST  origin 2026-04-01  cold 0.2216  satır 714688

 Kazanan modelde hata dağılımı (RMSLE^2 payları):
 profilli segment: satırların %78'i, hatanın %35'i
 cold-start segment: satırların %22'i, hatanın %65'i


### Profil dropout: cold-start segmentini eğitmek

Teşhis netti: hatanın %65'i cold-start segmentinden geliyor ve sebebi modelin egitim
sırasında profilsiz satırı yeterince görmemesi. Egitimde %1-14, testte %22.

Çözüm sub_01 -> sub_02'de işe yarayan prensibin aynısı: egitim dağılımını test dağılımına
benzet. Orada eksik olan yaz aylarıydı, burada eksik olan profilsiz satırlar.

Yöntem basit: egitim setinde rastgele seçilen trafoların profil sütunlarını NaN yapıyoruz.
Model böylece "geçmiş yokken grup istatistiğine yaslan" davranışını öğreniyor. Oranı
sabitlemek yerine tarayıp validationa seçtiriyoruz.

Dikkat: dropout bedava değil. Profili silinen trafo, profilli örnek olarak da kaybediliyor,
yani yüksek oranlarda profilli segment bozulmaya başlar. Aradığımız şey iki segmentin
toplamını en aza indiren nokta, o yüzden ikisini ayrı ayrı izliyoruz.

In [ ]:
# Kazanan konfigürasyonu sabit tutup sadece dropout oranını değiştiriyoruz
OZ, LR, LEAVES = kazanan_A['ozellikler'], kazanan_A['lr'], kazanan_A['leaves']

sonuclar_C = []
for oran in [0.0, 0.15, 0.25, 0.35, 0.50, 0.65, 0.80, 0.90]:
    eg = egitim_seti if oran == 0 else cold_start_uygula(egitim_seti, oran, tohum=7)
    gercek_oran = eg['log_ortalama'].isna().mean()
    dene(f'dropout {oran:.0%} (gercek {gercek_oran:.0%})', OZ, eg, val_seti, sonuclar_C, LR, LEAVES)

print()
kazanan_C = tablola(sonuclar_C, 'PROFIL DROPOUT taramasi (val cold-start orani %22)')
print()
print(f"referans (dropout yok): {sonuclar_C[0]['skor']:.4f}")
print(f"en iyi                : {kazanan_C['ad']} -> {kazanan_C['skor']:.4f}")
print(f"kazanc                : {sonuclar_C[0]['skor'] - kazanan_C['skor']:+.4f}")
print(f"cold-start segmentinde: {sonuclar_C[0]['cold']:.4f} -> {kazanan_C['cold']:.4f}")
print(f"profilli segmentinde  : {sonuclar_C[0]['profilli']:.4f} -> {kazanan_C['profilli']:.4f}")
print()
print(f"Kaggle öngörüsü (yerel - {OFFSET:.3f}): {kazanan_C['skor'] - OFFSET:.4f}   [mevcut en iyi 1.06168] --> bu şekilde olmasa da denemeye değer")

dropout 0% (gercek 1%)         RMSLE 1.2694  (profilli 0.8559 | cold-start 2.1668)  agac 58, ozellik 24
dropout 15% (gercek 15%)       RMSLE 1.2155  (profilli 0.9162 | cold-start 1.9279)  agac 46, ozellik 24
dropout 25% (gercek 25%)       RMSLE 1.2113  (profilli 0.9123 | cold-start 1.9225)  agac 46, ozellik 24
dropout 35% (gercek 35%)       RMSLE 1.1993  (profilli 0.9116 | cold-start 1.8894)  agac 51, ozellik 24
dropout 50% (gercek 50%)       RMSLE 1.1893  (profilli 0.9197 | cold-start 1.8465)  agac 51, ozellik 24
dropout 65% (gercek 65%)       RMSLE 1.1849  (profilli 0.9303 | cold-start 1.8150)  agac 49, ozellik 24
dropout 80% (gercek 80%)       RMSLE 1.1822  (profilli 0.9241 | cold-start 1.8181)  agac 60, ozellik 24
dropout 90% (gercek 90%)       RMSLE 1.2242  (profilli 0.9890 | cold-start 1.8237)  agac 70, ozellik 24

PROFIL DROPOUT taramasi (val cold-start orani %22)
                   deney  RMSLE  profilli   cold  agac  ozellik
dropout 80% (gercek 80%) 1.1822    0.9241 1.8181    

In [ ]:
print('kazanan:', kazanan_A['ad'], '-> RMSLE', round(kazanan_A['skor'], 4),
      '| profilli', round(kazanan_A['profilli'], 4), '| cold-start', round(kazanan_A['cold'], 4))
print('hava durumu dahil:', 'sic_ort' in kazanan_A['ozellikler'],
      '| yilin_gunu dahil:', 'yilin_gunu' in kazanan_A['ozellikler'])
print()

# Final egitimde sızıntı olamaz: test 2026 Nisan-Temmuz,  2025-2026 tüm veriyi kullanabiliriz
# Yaz origin'leri test aylarını kapsıyor, kış origin'i profili derinleştiriyor

final_egitim = pd.concat([origin_seti(bas, bit, bas) for bas, bit in FINAL_ORIGINLER],
                         ignore_index=True)
final_egitim['hedef_log'] = np.log1p(final_egitim['tuketim_duzeltilmis'])
for c in KATEGORIK:
    final_egitim[c] = final_egitim[c].astype('category')

test_seti = veri_seti_kur(train, test, pd.Timestamp('2026-04-01'))
for c in KATEGORIK:
    test_seti[c] = pd.Categorical(test_seti[c], categories=final_egitim[c].cat.categories)

print('final egitim:', final_egitim.shape, '| test:', test_seti.shape)
print('egitim ayları:', sorted(final_egitim['ay'].unique()))
print('cold-start oranı -> test:', round(test_seti['log_ortalama'].isna().mean(), 4))

kazanan: D4 lr=0.05 leaves=31 -> RMSLE 1.2694 | profilli 0.8559 | cold-start 2.1668
hava durumu dahil: False | yilin_gunu dahil: False

final egitim: (1306364, 44) | test: (714688, 41)
egitim ayları: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(12)]
cold-start oranı -> test: 0.2216


In [ ]:
sample_sub = pd.read_csv('../data/raw/sample_submission.csv')


def final_uret(dosya_adi, ayar, dropout=0.0):
    ozellikler = ayar['ozellikler']
    # validationda bulunan ağaç sayısı daha küçük bir eğitim setinden geldi, final set daha büyük pay bıraktık.
    agac = int(ayar['agac'] * 1.5)

    egitim = final_egitim if dropout == 0 else cold_start_uygula(final_egitim, dropout, tohum=7)

    model = lgb.LGBMRegressor(n_estimators=agac, learning_rate=ayar['lr'], num_leaves=ayar['leaves'], random_state=42, verbose=-1)
    model.fit(egitim[ozellikler], egitim['hedef_log'],
              categorical_feature=[c for c in KATEGORIK if c in ozellikler])

    tahmin = np.clip(np.expm1(model.predict(test_seti[ozellikler])), 0, None)
    sub = pd.DataFrame({'id': test_seti['id'], 'tuketim': tahmin})

    saglam = (len(sub) == len(sample_sub) and set(sub['id']) == set(sample_sub['id'])
              and sub['tuketim'].isna().sum() == 0 and (sub['tuketim'] < 0).sum() == 0)
    sub.to_csv(f'../submissions/{dosya_adi}.csv', index=False)

    print(f"{dosya_adi:26s} agac {agac:4d} lr {ayar['lr']:.2f} leaves {ayar['leaves']:3d} "
          f"dropout {dropout:.0%} | ortalama {sub['tuketim'].mean():7.1f} "
          f"medyan {sub['tuketim'].median():7.1f} max {sub['tuketim'].max():9.1f} "
          f"| doğrulama {'OK' if saglam else 'HATA'}")
    return model


# Validationun seçtiği dropout oranını final egitime uyguluyoruz.
# Final egitim zaten %7-14 doğal cold-start iceriyor, fonksiyon farkı tamamlıyor.
EN_IYI_DROPOUT = float(kazanan_C['ad'].split('%')[0].split()[-1]) / 100
print('validationun seçtiği dropout:', f'{EN_IYI_DROPOUT:.0%}')
print()

model_yeni = final_uret('sub_06_dropout', kazanan_A, dropout=EN_IYI_DROPOUT)

print()

onem = pd.DataFrame({'ozellik': kazanan_A['ozellikler'], 'onem': model_yeni.feature_importances_})
print('özellik önemi (ilk 15):')
print(onem.sort_values('onem', ascending=False).head(15).to_string(index=False))

validationun sectigi dropout: 80%

sub_06_dropout             agac   87 lr 0.05 leaves  31 dropout 80% | ortalama  1719.6 medyan  1129.5 max   44865.8 | doğrulama OK

özellik önemi (ilk 15):
         ozellik  onem
            ilce   576
             guc   477
              ay   371
   son30_log_ort   293
    grup_log_ort   277
           bolge   214
  trafo_ortalama    77
trafo_gun_sayisi    73
   son90_log_ort    50
    log_ortalama    42
             gun    37
    trafo_medyan    30
 trafo_gun_orani    22
       log_trend    21
         log_std    20


sub06 işe yaramadı: Your latest submission scored 1.10514. Your best score remains 1.06168.

